# SSLimPy — Observed Power Spectra

This notebook shows how to compute the observed (RSD + window-convolved) LIM
power spectra, scan over the parametric bias model `BAOpars`, and study the
Alcock–Paczynski and survey-resolution effects.

In [ ]:
import os
import sys

sys.path.append("../")

# Use the available cores for the Einstein--Boltzmann solver (CAMB/CLASS) and numba
os.environ.setdefault("OMP_NUM_THREADS", "12")

import astropy.units as u
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
Cs = sns.color_palette("colorblind")
import niceplots
niceplots.utils.initPlot()

## 1. Setup

In [ ]:
settings = {
    "code": "camb",
    "do_RSD": True,
    "nonlinearRSD": True,
    "QNLpowerspectrum": True,
    "FoG_damp": "ISTF_like",
    "halo_model_PS": False,
    "output": ["Power spectrum", "Covariance"],
    "kmin": 1e-4 * u.Mpc**-1,
    "kmax": 50 * u.Mpc**-1,
    "nk": 200,
}

cosmodict = {
    "h": 0.6737,
    "Omegam": 0.3146,
    "Omegab": 0.0492,
    "As": 2.1e-9,
    "ns": 0.966,
    "mnu": 0.06,
    "Neff": 3.044,
}

halodict = {
    "halo_tracer": "clustering",
    "hmf_model": "ST",
    "concentration": "Diemer19",
    "bias_model": "ST99",
    "bias_par": {"q": 0.707, "p": 0.3},
}

astrodict = {
    "model_type": "ML",
    "model_name": "SilvaCII",
    "model_par": {
        "a": 0.939,
        "b": 6.639,
        "SFR_file": "Fonseca16_Lya_SFR_params.dat",
        "do_quench": True,
    },
    "sigma_scatter": 0.5,
}

surveyspecs = {
    "Tsys_NEFD": 40 * u.uK,
    "Nfeeds": 19,
    "tobs": 1300 * u.h,
    "nD": 1,
    "beam_FWHM": 0.5 * u.arcmin,
    "nu": 1.897 * u.THz,        # [CII]
    "dnu": 900 * u.MHz,
    "nuObs": 250 * u.GHz,
    "Delta_nu": 50 * u.GHz,
    "Omega_field": 140 * u.deg**2,
}

In [ ]:
from SSLimPy.interface import sslimpy

myssl = sslimpy.SSLimPy(
    settings_dict=settings,
    cosmopars=cosmodict,
    halopars=halodict,
    astropars=astrodict,
    obspars_dict=surveyspecs,
)

## 2. Compute the power spectrum

The internal grids of the code are tuned for numerical stability; the observed
power spectrum is resampled on the grid of the survey via `pobs_settings`.
The *parametric* bias is passed in `BAOpars` (`bmean`, `Tmean`, `sigmap`,
`sigmav`, `alpha_iso`, `alpha_AP`, `beta`, `Pshot`).

In [ ]:
pobs_settings = {
    "kmin": 1e-3 * u.Mpc**-1,
    "kmax": 5 * u.Mpc**-1,
    "nk": 100,
    "k_kind": "log",
    "nmu": 128,
}

BAOdict = {"bmean": 2.8}

results = myssl.compute(
    myssl.fiducialcosmoparams,
    myssl.fiducialhaloparams,
    myssl.fiducialastroparams,
    myssl.fiducialspecparams,
    BAOpars=BAOdict,
    pobs_settings=pobs_settings,
    output=["Power spectrum"],
)
pobs = results["Power spectrum"]

## 3. Multipoles

`Pk_Obs` has shape `(nk, nmu, nz)`; the multipoles `Pk_0bs`, `Pk_2bs`, `Pk_4bs`
are already projected and have shape `(nk, nz)`.

In [ ]:
k = pobs.k

plt.loglog(k, k[:, None] * pobs.Pk_0bs, c=Cs[0], label=r"$\ell=0$")
plt.loglog(k, k[:, None] * pobs.Pk_2bs, c=Cs[1], label=r"$\ell=2$")
plt.loglog(k, -k[:, None] * pobs.Pk_2bs, c=Cs[1], ls="--")
plt.loglog(k, k[:, None] * pobs.Pk_4bs, c=Cs[2], label=r"$\ell=4$")
plt.loglog(k, -k[:, None] * pobs.Pk_4bs, c=Cs[2], ls="--")
plt.xlabel(r"$k\,[\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$k\,P_\ell(k)\,[\mu\mathrm{K}^2\,\mathrm{Mpc}^{2}]$")
plt.legend()

## 4. Scaling with the bias

In [ ]:
for b in [1.5, 2.5, 3.5]:
    res = myssl.compute(
        myssl.fiducialcosmoparams, myssl.fiducialhaloparams,
        myssl.fiducialastroparams, myssl.fiducialspecparams,
        BAOpars={"bmean": b}, pobs_settings=pobs_settings,
        output=["Power spectrum"],
    )["Power spectrum"]
    plt.loglog(k, k * res.Pk_0bs.squeeze(), label=r"$\langle b \rangle = {} $".format(b))

plt.xlabel(r"$k\,[\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$k\,P_0(k)\,[\mu\mathrm{K}^2\,\mathrm{Mpc}^{2}]$")
plt.legend()

## 5. Alcock–Paczynski effect

Rescaling the distances (`alpha_iso`, `alpha_AP`) shifts the BAO/`k` position
and distorts the quadrupole — degenerating with cosmology.

In [ ]:
color = iter(Cs)
for a in [1.0, 1.05, 1.1]:
    res = myssl.compute(
        myssl.fiducialcosmoparams, myssl.fiducialhaloparams,
        myssl.fiducialastroparams, myssl.fiducialspecparams,
        BAOpars={"bmean": 2.8, "alpha_iso": a}, pobs_settings=pobs_settings,
        output=["Power spectrum"],
    )["Power spectrum"]
    plt.loglog(k, k * res.Pk_0bs.squeeze(), label=r"$\alpha_{\rm iso} =" +"{:.2f}$".format(a), c=next(color))

plt.xlabel(r"$k\,[\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$k\,P_0(k)\,[\mu\mathrm{K}^2\,\mathrm{Mpc}^{2}]$")
plt.legend()

## 6. Survey resolution and window effects

The survey specifications carry the survey window $W(\mathbf q)$ and the
resolution suppression factors $F_{\parallel,\perp}(k,\mu)$ that are applied to
the power spectrum when `Smooth_window` / `Smooth_resolution` are set.

In [ ]:
specs = pobs.survey_specs

q = np.geomspace(1e-3, 5, 60) * u.Mpc**-1
muq = np.linspace(-1, 1, 21)
W = specs.Wsurvey(q, muq).squeeze()

plt.pcolormesh(muq, q.value, W.value, shading="nearest")
plt.colorbar(label=r"$W(\mathbf q)\,[\mathrm{Mpc}^3]$")
plt.yscale("log")
plt.xlabel(r"$\mu$")
plt.ylabel(r"$q\,[\mathrm{Mpc}^{-1}]$")

In [ ]:
Fp = specs.F_parr(k, pobs.mu).squeeze()
Fs = specs.F_perp(k, pobs.mu).squeeze()

plt.loglog(k, Fp[:, 0], c=Cs[0], label=r"$F_\parallel\,(\mu=0)$")
plt.loglog(k, Fp[:, -1], c=Cs[1], ls="--", label=r"$F_\parallel\,(\mu=1)$")
plt.loglog(k, Fs[:, 0], c=Cs[2], ls=":", label=r"$F_\perp\,(\mu=0)$")
plt.xlabel(r"$k\,[\mathrm{Mpc}^{-1}]$")
plt.ylabel("resolution suppression")
plt.legend()